<a href="https://colab.research.google.com/github/fisherynwa/adv-rag-benchmark/blob/main/test_retest_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Test-retest: is the result reproducible?

This notebook loads the best RAG system exported from the Kaggle benchmark (`best_rag.zip`) and asks two questions:

1. **Is the judge consistent?** The judge scores the *same* Kaggle answers again. If it is reproducible, it gives
   the same verdicts.
2. **Is the system consistent?** The exported system answers all 154 questions again, on a different machine (a Colab
   GPU instead of Kaggle's), and the judge scores the new answers.

**How to run:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. The second cell asks you to
upload `best_rag.zip` from the Kaggle notebook's output. It takes roughly 30 minutes. Keep the tab open, since Colab
stops idle sessions.

In [ ]:
# 1. Setup: install packages, start Ollama, load the same models as on Kaggle
import subprocess, sys, time, json, re, logging, shutil, urllib.request, zipfile, os

def sh(cmd, label):              # run quietly; show the output only if it fails
    print(label + "...")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{label} failed:\n{(r.stdout + r.stderr)[-2000:]}")

sh(f"{sys.executable} -m pip install -q llama-index-core==0.14.25 llama-index-llms-ollama==0.11.0 "
   "llama-index-embeddings-ollama==0.10.0 nest_asyncio", "Installing Python packages")
if shutil.which("ollama") is None:
    sh("apt-get update -qq && apt-get install -y -qq zstd", "Installing zstd")
    sh("curl -fsSL https://ollama.com/install.sh | sh", "Installing Ollama")
subprocess.Popen("OLLAMA_CONTEXT_LENGTH=24576 OLLAMA_NUM_PARALLEL=1 OLLAMA_KEEP_ALIVE=-1 ollama serve > ollama.log 2>&1",
                 shell=True)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("The Ollama server did not start:\n" + open("ollama.log").read()[-2000:])

import nest_asyncio; nest_asyncio.apply()
logging.getLogger().setLevel(logging.WARNING)
for _name in ("httpx", "httpcore", "llama_index"):
    logging.getLogger(_name).setLevel(logging.WARNING)
import numpy as np, pandas as pd

Installing Python packages...
Installing zstd...
Installing Ollama...


In [ ]:
# 2. Load the exported system (upload best_rag.zip from the Kaggle output)
if not os.path.exists("best_rag.zip"):
    from google.colab import files
    files.upload()                                  # choose best_rag.zip
zipfile.ZipFile("best_rag.zip").extractall("best_rag")
cfg = json.load(open("best_rag/config.json"))
key = json.load(open("best_rag/answer_key.json"))["questions"]
kaggle = pd.read_csv("best_rag/kaggle_results.csv")
kaggle["answer"] = kaggle["answer"].fillna("").astype(str)
questions_by_id = {q["id"]: q for q in key}

for m in (cfg["llm"], cfg["embed_model"]):
    sh(f"ollama pull {m}", f"Downloading {m}")
out = lambda cmd: subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
RUN_INFO = {"platform": "colab", "gpu": out("nvidia-smi --query-gpu=name --format=csv,noheader"),
            "ollama": out("ollama --version"), "models": out("ollama list")}
print(f"System: {cfg['system']} | {len(kaggle)} Kaggle answers")
print(f"Kaggle: {cfg['run_info']['gpu']} | {cfg['run_info']['ollama']}")
print(f"Colab:  {RUN_INFO['gpu']} | {RUN_INFO['ollama']}")

Saving best_rag.zip to best_rag.zip
System: sentence window 5 | 154 Kaggle answers
Kaggle: Tesla T4
Tesla T4 | ollama version is 0.40.7
Colab:  Tesla T4 | ollama version is 0.40.8


In [ ]:
# 3. Rebuild the query engine from the saved index and settings
from llama_index.core import StorageContext, load_index_from_storage, PromptTemplate
from llama_index.core.postprocessor import MetadataReplacementPostProcessor
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

llm = Ollama(model=cfg["llm"], context_window=24576, request_timeout=600, temperature=0, thinking=False)
embed = OllamaEmbedding(model_name=cfg["embed_model"], query_instruction="search_query: ",
                        text_instruction="search_document: ")       # must match how the index was built
index = load_index_from_storage(StorageContext.from_defaults(persist_dir="best_rag/index"), embed_model=embed)
qa_prompt = PromptTemplate(cfg["qa_prompt"])
if cfg["kind"] == "sentence_window":
    engine = index.as_query_engine(llm=llm, similarity_top_k=cfg["top_k"], text_qa_template=qa_prompt,
                                   node_postprocessors=[MetadataReplacementPostProcessor(target_metadata_key="window")])
elif cfg["kind"] == "auto_merging":
    retriever = AutoMergingRetriever(index.as_retriever(similarity_top_k=cfg["top_k"]), index.storage_context)
    engine = RetrieverQueryEngine.from_args(retriever, llm=llm, text_qa_template=qa_prompt)
else:
    engine = index.as_query_engine(llm=llm, similarity_top_k=cfg["top_k"], text_qa_template=qa_prompt)
print("Engine ready:", cfg["kind"])

Engine ready: sentence_window


In [ ]:
# 4. The judge, with the same fix as on Kaggle
from llama_index.core.evaluation import CorrectnessEvaluator
correctness = CorrectnessEvaluator(llm=llm)

def judge(evaluate, field="passing"):    # unreadable replies count as missing; fresh async client per call
    if hasattr(llm, "_async_client"):
        llm._async_client = None
    try:
        value = getattr(evaluate(), field)
        return np.nan if value is None else float(value)
    except Exception:
        return np.nan

def correct(question_id, answer):
    q = questions_by_id[question_id]
    return judge(lambda: correctness.evaluate(query=q["question"], response=answer, reference=q["answer"]))

## Check 1: is the judge consistent?

The judge scores the Kaggle answers again. **Agreement** is the share of answers with the same verdict as on
Kaggle; **Cohen's kappa** corrects for agreement by chance (above 0.8 is very consistent).

In [ ]:
t0 = time.time()
kaggle["correct_again"] = [correct(r.id, r.answer) for r in kaggle.itertuples()]
both = kaggle.dropna(subset=["correct", "correct_again"])

def agreement(a, b):
    a, b = np.asarray(a, int), np.asarray(b, int)
    po = (a == b).mean()
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return po, ((po - pe) / (1 - pe) if pe < 1 else float("nan"))

po, kappa = agreement(both.correct, both.correct_again)
print(f"Judge re-scored {len(both)} answers in {(time.time() - t0) / 60:.0f} min: "
      f"same verdict on {po:.0%}, Cohen's kappa {kappa:.2f}")
print(f"Share correct: Kaggle judge {both.correct.mean():.2f}, Colab judge {both.correct_again.mean():.2f}")
for r in both[both.correct != both.correct_again].head(5).itertuples():
    print(f"  changed [{r.type}] {questions_by_id[r.id]['question'][:80]} -> {r.correct:.0f} then {r.correct_again:.0f}")

Judge re-scored 154 answers in 10 min: same verdict on 100%, Cohen's kappa 1.00
Share correct: Kaggle judge 0.93, Colab judge 0.93


## Check 2: is the system consistent?

The exported system answers every question again, and the judge scores the new answers. **Identical answers** shows
how often the model wrote exactly the same text; the paired difference shows whether accuracy changed.

In [ ]:
t0, rows = time.time(), []
for q in key:
    answer = str(engine.query(q["question"]))
    rows.append({"id": q["id"], "answer_colab": answer, "correct_colab": correct(q["id"], answer)})
colab = kaggle.merge(pd.DataFrame(rows), on="id")
colab.to_csv("test_retest_results.csv", index=False)

rng = np.random.default_rng(0)
def ci(values):
    v = np.asarray(values, float); v = v[~np.isnan(v)]
    means = [rng.choice(v, len(v)).mean() for _ in range(2000)]
    return v.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

same_text = (colab.answer.str.strip() == colab.answer_colab.str.strip()).mean()
paired = colab.dropna(subset=["correct", "correct_colab"])
m, lo, hi = ci(paired.correct_colab - paired.correct)
print(f"Answered and judged {len(colab)} questions in {(time.time() - t0) / 60:.0f} min")
print(f"Identical answers: {same_text:.0%}")
print(f"Share correct: Kaggle {paired.correct.mean():.2f}, Colab {paired.correct_colab.mean():.2f} "
      f"(difference {m:+.2f}, 95% interval {lo:+.2f} to {hi:+.2f}, n={len(paired)})")

Answered and judged 154 questions in 19 min
Identical answers: 100%
Share correct: Kaggle 0.93, Colab 0.93 (difference +0.00, 95% interval +0.00 to +0.00, n=154)
